# 01. PyTorch, As Is
© 2026, Anyscale. All Rights Reserved

This notebook trains a ResNet18 on MNIST with a plain PyTorch loop, the same one in `src/train_torch.py`. Nothing here is Ray Train yet. The one Ray idea this notebook uses is a plain Ray task, to solve a very concrete problem: this notebook's kernel runs on the head node, and the head node has no GPU.

<div class="alert alert-block alert-info">

<b> Here is the roadmap for this notebook </b>

<ol>
  <li>Write the training loop</li>
  <li>Build the model and load the data</li>
  <li>Look at the metrics and checkpoint helper</li>
  <li>Find out where this can actually run</li>
  <li>Run the unchanged loop on a GPU worker</li>
  <li>Run the identical code as a script</li>
  <li>Use the checkpoint to make predictions</li>
  <li>Activity: fine-tune for one more epoch</li>
  <li>What notebook 02 replaces</li>
</ol>

</div>

**Imports**

In [ ]:
import dataclasses
import datetime
import inspect
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ray
import torch

## 1. Write the training loop

This repo keeps every stage of training as both a script and a notebook, so there is exactly one place each piece of code lives. Import the real thing from `src/train_torch.py` rather than retyping it, and print its source so you can read every line here:

In [ ]:
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
print(f"notebook cwd: {Path.cwd()}")
print(f"REPO_ROOT:    {REPO_ROOT}")

from src.train_torch import save_checkpoint_and_metrics, train_loop_torch, train_one_epoch

print(inspect.getsource(train_loop_torch))
print(inspect.getsource(train_one_epoch))

|<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-ai-libraries/diagrams/single_gpu_pytorch_v3.png" width="70%" loading="lazy">|
|:--|
|Single-GPU training: load the dataset, run mini-batches through the model on one GPU, checkpoint to persistent storage. `train_loop_torch` and `train_one_epoch` above are exactly this, written down.|

<div class="alert alert-block alert-info">

<b>Three lines here assume a single machine</b>

<ul>
  <li><code>model = build_resnet18().to(device)</code> in <code>train_loop_torch</code> puts the whole model on one device, chosen by hand.</li>
  <li><code>images, labels = images.to(device), labels.to(device)</code> in <code>train_one_epoch</code> moves every batch to that same device, also by hand.</li>
  <li>the <code>DataLoader</code> that <code>build_data_loader</code> returns hands the <b>entire</b> dataset to this one process. Nothing shards it across workers.</li>
</ul>

Starting in notebook 02, Ray Train makes these three lines work, unchanged, across many devices at once.

</div>

## 2. Build the model and load the data

`build_resnet18` lives in `src/model.py`:

In [ ]:
from src.model import build_resnet18

print(inspect.getsource(build_resnet18))

<div class="alert alert-block alert-info">

<code>resnet18</code>'s first conv layer, <code>model.conv1</code>, expects <code>in_channels=3</code> by default, one per RGB channel. MNIST digits are grayscale, a single channel, so <code>build_resnet18</code> replaces that layer with an equivalent one built for <code>in_channels=1</code>.

</div>

`src/data.py` has a `raw_mnist` helper built for exactly this: PIL images and plain labels, no tensor transform, meant for plotting. Load it and look at nine random examples:

In [ ]:
from src.settings import Settings

settings = Settings.from_env()

from src.data import raw_mnist

dataset = raw_mnist(settings.data_root)

In [ ]:
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3

for i in range(1, cols * rows + 1):
    sample_idx = np.random.randint(0, len(dataset.data))
    img, label = dataset[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(label)
    plt.axis("off")
    plt.imshow(img, cmap="gray")

Training itself needs tensors, normalized, batched, and shuffled, which is what `build_data_loader` returns:

In [ ]:
from src.data import build_data_loader

print(inspect.getsource(build_data_loader))

`settings.subset_size` is `None` by default and `2048` under `SMOKE_TEST=1`. Passing it straight through to `build_data_loader` is what makes a smoke test fast without touching this function at all.

## 3. Look at the metrics and checkpoint helper

`train_torch.py` doesn't give metrics reporting its own function, it folds that into the loop itself: build a small dict, print it. Checkpointing does get a dedicated helper, since it always does the same two things regardless of what's in `metrics`, append a row to `metrics.csv` and overwrite `model.pt`:

In [ ]:
print(inspect.getsource(save_checkpoint_and_metrics))

## 4. Find out where this can actually run

In [ ]:
print(f"torch.cuda.is_available() on the head: {torch.cuda.is_available()}")

That's `False`. On a laptop with a GPU, you would just call `train_loop_torch(settings, output_dir, "cuda")` right here and be done. On this cluster, every GPU lives on a worker node, never the head, so there are two honest ways to actually reach one:

1. Ask Ray to run this exact, unchanged function on a GPU worker: a `@ray.remote(num_gpus=1)` task.
2. Run `src/train_torch.py` as a script, submitted to a cluster whose compute config has a GPU. Section 6 below shows the same script running here in CPU mode as a preview; a real GPU submission is an Anyscale Job, which notebook 04 covers.

This notebook runs the first one.

## 5. Run the unchanged loop on a GPU worker

In [ ]:
from src.settings import ray_init_with_repo

ray_init_with_repo()

In [ ]:
stamp = datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d_%H-%M-%S")
output_dir = os.path.join(settings.storage_path, "01_pytorch_notebook", stamp)


@ray.remote(num_gpus=1)
def train_on_gpu_worker():
    import socket

    print(f"training on {socket.gethostname()} / {torch.cuda.get_device_name(0)}")
    metrics = train_loop_torch(settings, output_dir, "cuda")
    return socket.gethostname(), torch.cuda.get_device_name(0), metrics


hostname, gpu_name, metrics = ray.get(train_on_gpu_worker.remote(), timeout=3600)
print(f"host={hostname} gpu={gpu_name}")
print(f"output_dir={output_dir}")
print(f"final metrics={metrics}")

<div class="alert alert-block alert-info">

<code>output_dir</code> sits under <code>settings.storage_path</code>, shared cluster storage, not the GPU worker's own local disk. The autoscaler can reclaim that worker the instant this task finishes, and its local disk disappears with it. Writing the checkpoint to storage every node can reach is the only way anything, including this notebook's own driver, can read it back afterward. It's the same reason Ray Train's <code>RunConfig</code> takes a <code>storage_path</code>: notebook 02 hits this exact requirement again, just with more than one worker writing at once.

</div>

In [ ]:
!ls -l {output_dir}

In [ ]:
metrics_df = pd.read_csv(os.path.join(output_dir, "metrics.csv"), header=None, names=["epoch", "loss"])
metrics_df

## 6. Run the identical code as a script

The same function also exists as a file. Force CPU mode so it runs right here on the head, with a small data subset, no Ray involved at all, just to see the same code work as a script:

In [ ]:
!cd {REPO_ROOT} && SMOKE_TEST=1 USE_GPU=0 python src/train_torch.py

A real run would drop `SMOKE_TEST=1` and `USE_GPU=0`, and submit this same script to a cluster with a GPU compute config as an Anyscale Job, which notebook 04 covers.

## 7. Use the checkpoint to make predictions

Load the state dict straight from shared storage. The head has no GPU, so predictions run on the CPU here, that's fine, `model.pt` doesn't care which device wrote it:

In [ ]:
device = "cpu"
loaded_model = build_resnet18()
loaded_model.load_state_dict(torch.load(os.path.join(output_dir, "model.pt"), map_location=device))
loaded_model.to(device)
loaded_model.eval()

In [ ]:
from src.data import MNIST_TRANSFORM

figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3

for i in range(1, cols * rows + 1):
    sample_idx = np.random.randint(0, len(dataset.data))
    img, label = dataset[sample_idx]
    normalized_img = MNIST_TRANSFORM(img).to(device)

    with torch.no_grad():
        prediction = loaded_model(normalized_img.unsqueeze(0)).argmax().cpu()

    figure.add_subplot(rows, cols, i)
    color = "black" if int(prediction) == label else "red"
    plt.title(f"label: {label}; pred: {int(prediction)}", color=color)
    plt.axis("off")
    plt.imshow(img, cmap="gray")

Correct predictions are titled in black, wrong ones in red.

## 8. Activity: fine-tune for one more epoch

<div class="alert alert-block alert-info">

<b>Train one more epoch, at one tenth the learning rate, on a GPU worker, into a new output directory. Compare its final loss against the run in Section 5.</b>

Everything you need is already in scope: <code>settings</code>, <code>train_loop_torch</code>, and the same <code>@ray.remote(num_gpus=1)</code> pattern from Section 5. `dataclasses.replace` gives you a modified copy of `settings` without touching the original.

```python
# Hint
finetune_settings = dataclasses.replace(settings, lr=..., num_epochs=...)
```

</div>

In [ ]:
# Write your solution here


<div class="alert alert-block alert-info">

<details>

<summary> Click here to see the solution </summary>

```python
finetune_settings = dataclasses.replace(settings, lr=settings.lr / 10, num_epochs=1)
finetune_stamp = datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d_%H-%M-%S")
finetune_output_dir = os.path.join(settings.storage_path, "01_pytorch_notebook_finetune", finetune_stamp)


@ray.remote(num_gpus=1)
def finetune_on_gpu_worker():
    return train_loop_torch(finetune_settings, finetune_output_dir, "cuda")


finetune_metrics = ray.get(finetune_on_gpu_worker.remote(), timeout=3600)
print(f"previous final loss (Section 5): {metrics['loss']:.4f}")
print(f"finetune final loss:             {finetune_metrics['loss']:.4f}")
```

One epoch at a tenth of the original learning rate rarely swings the loss by much either way, this is a check that fine-tuning from a fresh `Settings` works, not a claim that it always improves things.

</details>

</div>

## 9. What notebook 02 replaces

The loop above works, checkpoints correctly, and produced a real prediction grid. It also only ever uses one GPU, because two things in it assume exactly one device exists:

- Every `.to(device)` call, on the model and on each batch, picks that one device by hand.
- The plain `DataLoader` from `build_data_loader` hands the whole dataset to whichever process calls it, no sharding.

Next: notebook 02 replaces both, using `ray.train.torch.prepare_model` and `ray.train.torch.prepare_data_loader`, without changing the training logic itself.

## Further reading

| Resource | Why it matters here |
|---|---|
| [PyTorch training loop basics](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) | the plain `train_loop_torch` loop from section 1 |
| [Saving and loading models](https://pytorch.org/tutorials/beginner/saving_loading_models.html) | the `model.pt` checkpoint written in section 5 |
| [torch.export](https://pytorch.org/docs/stable/export.html) | an alternative to the state-dict checkpoint used in section 5 |
| [Local storage](https://docs.anyscale.com/storage/local) | why the GPU worker's own disk disappears, per section 5's callout |
| [Shared storage](https://docs.anyscale.com/storage/shared) | the `settings.storage_path` every checkpoint here writes to |
| [Ray tasks](https://docs.ray.io/en/latest/ray-core/tasks.html) | the `@ray.remote(num_gpus=1)` pattern used to reach the GPU worker |
| [Template: pytorch-profiling](https://github.com/anyscale/templates/tree/main/templates/pytorch-profiling) | profiling the same kind of loop this notebook trains |